Databricks notebook: nb_gold_dim_station
Source : ev_catalog.silver.sl_stations  +  sl_cities  +  sl_states
Sink   : ev_catalog.gold.dim_station  (Delta, external)
Type   : SCD Type 1 — overwrite changed attributes in-place
Grain  : One row per physical charging station

Silver columns used:
  sl_stations : station_id, name, state_code, city, latitude, longitude,
                charger_type, max_power_kw, num_connectors, operator, is_active, commissioned_date
  sl_cities   : city_id, city_name, state_code, postcode
  sl_states   : state_code, state_name

In [ ]:
# ── Cell 1: imports and config ────────────────────────────────────────────────
from pyspark.sql import functions as F
from delta.tables import DeltaTable

GOLD_TABLE = "ev_catalog.gold.dim_station"
GOLD_PATH  = "abfss://gold@evdatalakedev2.dfs.core.windows.net/dimensions/dim_station/"

dbutils.widgets.text("p_run_date", "", "Run date YYYY-MM-DD (blank = full load)")
run_date = dbutils.widgets.get("p_run_date").strip()

In [ ]:
# ── Cell 2: read Silver sources ────────────────────────────────────────────────
df_stations = spark.table("ev_catalog.silver.sl_stations")
df_cities   = spark.table("ev_catalog.silver.sl_cities")
df_states   = spark.table("ev_catalog.silver.sl_states")

In [ ]:
# ── Cell 3: join and build dimension ──────────────────────────────────────────
# sl_stations.city  is a city-name string (not a foreign key id).
# Join to sl_cities on city name to get city_id and postcode.
# Join to sl_states on state_code to get full state_name.

df_dim = (
    df_stations.alias("st")
    .join(
        df_cities.alias("ci"),
        F.col("st.city") == F.col("ci.city_name"),
        "left"
    )
    .join(
        df_states.alias("sr"),
        F.col("st.state_code") == F.col("sr.state_code"),
        "left"
    )
    .select(
        F.col("st.station_id")                        .alias("station_key"),
        F.col("st.name")                              .alias("station_name"),
        F.col("st.charger_type"),
        F.col("st.max_power_kw").cast("double"),
        F.col("st.num_connectors").cast("int"),
        F.col("st.operator"),
        F.col("st.is_active"),
        F.col("st.latitude").cast("double"),
        F.col("st.longitude").cast("double"),
        F.col("st.commissioned_date"),
        F.col("ci.city_id")                           .alias("city_key"),
        F.col("ci.city_name"),
        F.col("ci.postcode"),
        F.col("st.state_code"),
        F.col("sr.state_name"),
        F.current_timestamp()                         .alias("gold_updated_at"),
    )
)

print(f"dim_station rows: {df_dim.count()}")
df_dim.show(3, truncate=80)

In [ ]:
# ── Cell 4: CREATE TABLE IF NOT EXISTS ────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
    station_key        STRING   NOT NULL,
    station_name       STRING,
    charger_type       STRING,
    max_power_kw       DOUBLE,
    num_connectors     INT,
    operator           STRING,
    is_active          BOOLEAN,
    latitude           DOUBLE,
    longitude          DOUBLE,
    commissioned_date  STRING,
    city_key           STRING,
    city_name          STRING,
    postcode           STRING,
    state_code         STRING,
    state_name         STRING,
    gold_updated_at    TIMESTAMP
)
USING DELTA
LOCATION '{GOLD_PATH}'
""")

In [ ]:
# ── Cell 5: SCD1 MERGE (overwrite in place) ───────────────────────────────────
tgt = DeltaTable.forName(spark, GOLD_TABLE)
(
    tgt.alias("t")
       .merge(df_dim.alias("s"), "t.station_key = s.station_key")
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll()
       .execute()
)
print(f"MERGE done: {GOLD_TABLE}")

In [ ]:
# ── Cell 6: verify ────────────────────────────────────────────────────────────
spark.sql(f"SELECT state_name, COUNT(*) AS station_count FROM {GOLD_TABLE} GROUP BY state_name ORDER BY station_count DESC").display()